# UUIDs and the Default Factory Trap

Universally Unique Identifiers (UUIDs) are standard data types used heavily in modern APIs and databases to uniquely identify records. While Python has a built-in `uuid` module to generate them, Pydantic provides specialized types to validate and parse them effortlessly.

## 1. Using Pydantic's UUID Types

Pydantic has built-in types for different versions of UUIDs (e.g., `UUID1`, `UUID3`, `UUID4`, `UUID5`). The most common for generating random unique IDs is `UUID4`.

When you type hint a field with `UUID4`, Pydantic will automatically validate incoming data to ensure it is a valid UUID4, and it will automatically coerce standard UUID strings into native Python `UUID` objects.

```python
from uuid import uuid4
from pydantic import BaseModel, Field, UUID4, ValidationError

class Person(BaseModel):
    id_: UUID4 = Field(alias="id")
    name: str

# You can instantiate it using a native Python UUID object
p1 = Person(id=uuid4(), name="Alice")

# Or you can instantiate it using a raw JSON string
p2 = Person.model_validate_json('{"id": "123e4567-e89b-12d3-a456-426614174000", "name": "Bob"}')

# Pydantic parses the string into a native Python UUID object automatically!
print(type(p2.id_)) # <class 'uuid.UUID'>

```

### Serialization Behavior

Just like `datetime` objects:

* **`model_dump()`:** Returns a dictionary where the value is the native Python `UUID` object.
* **`model_dump_json()`:** Automatically converts the native `UUID` object into a standard string representation for JSON compatibility.

## 2. The Default Value Trap (Crucial)

If you want to auto-generate a UUID for a model if the user doesn't provide one, your first instinct might be to set the default value using the `uuid4()` function.

```python
# ❌ INCORRECT WAY - THE DEFAULT TRAP
class BuggyPerson(BaseModel):
    # Trap 1: Standard assignment
    id_: UUID4 = uuid4() 
    
    # Trap 2: Using the Field default
    id_alias: UUID4 = Field(alias="id", default=uuid4())

```

**The Problem:** Both of these implementations have a massive bug. If you create two instances (`p1 = BuggyPerson()`, `p2 = BuggyPerson()`), **they will both have the exact same UUID.**

**The Cause:** In Python, class attributes and default function arguments are evaluated exactly once: when the script is first run and the class is compiled. Python executes the `uuid4()` function during compilation, generates *one* UUID, and assigns that static value as the default for every single instance created afterward.

## 3. The Solution: `default_factory`

To fix this, we must tell Pydantic to execute the function dynamically *every single time* a new instance is created. We do this by passing the function reference (without the parentheses) to the `default_factory` parameter inside the `Field` object.

```python
# ✅ CORRECT WAY - USING DEFAULT FACTORY
class SafePerson(BaseModel):
    # Notice: uuid4, NOT uuid4()
    id_: UUID4 = Field(alias="id", default_factory=uuid4)
    name: str

p1 = SafePerson(name="Alice")
p2 = SafePerson(name="Bob")

print(p1.id_) # e.g., 550e8400-e29b-41d4-a716-446655440000
print(p2.id_) # e.g., 8a12b322-811c-4b35-8625-f3776b71f927
# The UUIDs are uniquely generated for each instance!

```

*Note: `default_factory` is universally useful. You must use it anytime you want a dynamic default value, such as `datetime.now`, `list`, or `dict`.*



### Interview Preparation: UUIDs & Default Factories

<details>
<summary><b>1. You build a `User` model with `created_at: datetime = datetime.now()` and `id: UUID4 = uuid4()`. In production, you notice that every user created in the database today has the exact same ID and timestamp. What caused this bug and how do you fix it?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
This is a classic Python evaluation trap. Because class attributes are evaluated at compile time (when the module is imported), Python called `uuid4()` and `datetime.now()` exactly once when the application started, and permanently bound those static values as the defaults for all future instances of the class. <br><br>
To fix this, you must use Pydantic's `default_factory`. You pass the function reference (without calling it) so Pydantic executes it dynamically upon every object instantiation: <br>
`created_at: datetime = Field(default_factory=datetime.now)`<br>
`id: UUID4 = Field(default_factory=uuid4)`
</details>

<details>
<summary><b>2. An API consumer sends a JSON payload containing `{"user_id": "a0eebc99-9c0b-4ef8-bb6d-6bb9bd380a11"}`. The Pydantic model types this field as `user_id: UUID4`. When you access `model.user_id` in your Python code, what is the underlying data type, and why?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
The underlying data type is a native Python `uuid.UUID` object. <br><br>
Pydantic's specialized types not only validate that the string matches the strict format of a UUID version 4, but they also act as parsers. During deserialization, Pydantic coerces the raw JSON string into the native Python object automatically, allowing you to use native UUID methods throughout your backend code.
</details>

<details>
<summary><b>3. Why is it impossible to define `alias="id"` and a dynamic UUID generation fallback simply by using standard Python assignment (e.g., `id_: UUID4 = uuid4()`)?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
Standard Python assignment on a class attribute assigns both the type hint and the static default value. It does not provide any syntax to attach metadata like an `alias`, nor does it provide a mechanism to execute a callable dynamically per instance. <br><br>
To attach configuration metadata (like `alias="id"`) while also providing a dynamic default, you are forced to use the right-hand assignment for the Pydantic `Field` object. Once using `Field`, you map the dynamic callable to the `default_factory` parameter.
</details>